In [0]:
spark.sql("""
CREATE OR REPLACE VIEW ledgr.gold.sessions_analyst_view AS
SELECT
    task_id,
    run_id,
    harness,
    benchmark,
    model_request,
    provider,
    outcome_state,
    execution_cost_usd,
    is_synthetic_retry
FROM ledgr.silver.calls_enriched
""")
print("Analyst view rebuilt on Silver")

In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("ledgr.silver.calls_enriched")

# Both variants of the core metric, computed at the model+harness grain
gold_cost_per_success = (
    silver_df.groupBy("model_request", "harness")
    .agg(
        F.sum("execution_cost_usd").alias("total_cost_usd"),
        F.sum(F.when(F.col("outcome_state") == "SUCCESS", F.col("execution_cost_usd")).otherwise(0)).alias("successful_cost_usd"),
        F.count(F.when(F.col("outcome_state") == "SUCCESS", 1)).alias("successful_calls"),
        F.count("*").alias("total_calls"),
        F.sum(F.when(F.col("is_synthetic_retry") == True, F.col("execution_cost_usd")).otherwise(0)).alias("wasted_retry_cost_usd"),
    )
    .withColumn("cost_per_successful_outcome_including_waste", 
        F.round(F.col("total_cost_usd") / F.col("successful_calls"), 6))
    .withColumn("cost_per_successful_outcome_success_only",
        F.round(F.col("successful_cost_usd") / F.col("successful_calls"), 6))
)

gold_cost_per_success.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("ledgr.gold.mart_cost_per_success")

print("Gold mart created")
gold_cost_per_success.orderBy(F.desc("total_cost_usd")).show(20, truncate=False)